# 🏠 House Price Prediction - ML Model
**Goal:** Predict house prices based on features like area, bedrooms, bathrooms, etc.

**Dataset:** Housing.csv (545 records, 13 features)

## 1️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully ✅')

## 2️⃣ Load & Explore the Data

In [ ]:
df = pd.read_csv('Housing.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum())

## 3️⃣ Exploratory Data Analysis (EDA)

In [ ]:
# Price Distribution
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.histplot(df['price'], kde=True, color='steelblue')
plt.title('House Price Distribution')
plt.xlabel('Price')

plt.subplot(1, 2, 2)
sns.histplot(np.log(df['price']), kde=True, color='coral')
plt.title('Log Price Distribution')
plt.xlabel('Log Price')

plt.tight_layout()
plt.show()

In [ ]:
# Area vs Price
plt.figure(figsize=(8, 5))
sns.scatterplot(x='area', y='price', data=df, alpha=0.6, color='steelblue')
plt.title('Area vs Price')
plt.xlabel('Area (sq ft)')
plt.ylabel('Price')
plt.show()

In [ ]:
# Price by number of bedrooms
plt.figure(figsize=(8, 5))
sns.boxplot(x='bedrooms', y='price', data=df, palette='Blues')
plt.title('Price by Number of Bedrooms')
plt.show()

In [ ]:
# Price by furnishing status
plt.figure(figsize=(8, 5))
sns.boxplot(x='furnishingstatus', y='price', data=df, palette='Set2')
plt.title('Price by Furnishing Status')
plt.show()

In [ ]:
# Correlation heatmap (numeric columns only)
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## 4️⃣ Data Preprocessing

In [ ]:
df_model = df.copy()

# Encode yes/no columns
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in binary_cols:
    df_model[col] = df_model[col].map({'yes': 1, 'no': 0})

# Encode furnishingstatus
df_model['furnishingstatus'] = df_model['furnishingstatus'].map({
    'furnished': 2,
    'semi-furnished': 1,
    'unfurnished': 0
})

print('Encoding done ✅')
df_model.head()

In [ ]:
# Features and Target
X = df_model.drop('price', axis=1)
y = df_model['price']

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

## 5️⃣ Train Multiple Models & Compare

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2 Score': r2}
    print(f'{name}')
    print(f'  MAE:      {mae:,.0f}')
    print(f'  RMSE:     {rmse:,.0f}')
    print(f'  R² Score: {r2:.4f}')
    print()

In [ ]:
# Compare R2 Scores visually
r2_scores = {name: results[name]['R2 Score'] for name in results}

plt.figure(figsize=(8, 5))
bars = plt.bar(r2_scores.keys(), r2_scores.values(), color=['steelblue', 'coral', 'seagreen'])
plt.title('Model Comparison — R² Score')
plt.ylabel('R² Score')
plt.ylim(0, 1)
for bar, val in zip(bars, r2_scores.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center')
plt.tight_layout()
plt.show()

## 6️⃣ Best Model — Deep Dive (Random Forest)

In [ ]:
best_model = models['Random Forest']
best_preds = best_model.predict(X_test)

# Actual vs Predicted
plt.figure(figsize=(8, 5))
plt.scatter(y_test, best_preds, alpha=0.6, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted Price (Random Forest)')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
feat_imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
feat_imp.plot(kind='bar', color='steelblue')
plt.title('Feature Importance — What Drives House Prices?')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7️⃣ Predict on a New House 🏠

In [ ]:
# Try predicting price for a new house
new_house = pd.DataFrame([{
    'area': 5000,
    'bedrooms': 3,
    'bathrooms': 2,
    'stories': 2,
    'mainroad': 1,
    'guestroom': 0,
    'basement': 1,
    'hotwaterheating': 0,
    'airconditioning': 1,
    'parking': 1,
    'prefarea': 0,
    'furnishingstatus': 1   # semi-furnished
}])

predicted_price = best_model.predict(new_house)[0]
print(f'🏠 Predicted Price: ₹{predicted_price:,.0f}')

## ✅ Summary

| Model | R² Score |
|-------|----------|
| Linear Regression | ~0.65 |
| Random Forest | ~0.85 |
| Gradient Boosting | ~0.83 |

**Key Findings:**
- `area` is the strongest predictor of house price
- `airconditioning` and `bathrooms` also significantly impact price
- Random Forest outperforms all models with ~85% R² score
